### **Imports**

In [0]:
import pyspark.sql.functions as F

import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
class DataVisualization:
    def __init__(self, source_catalog, source_schema, source_table_name):
        """
        Constructor to initialize the instance variables — variables that are needed to get data from the gold schema.
        """
        self.source_catalog = source_catalog
        self.source_schema = source_schema
        self.source_table_name = source_table_name

    def read_feature_table(self):
        """
        Function to read the Gold Feature Table using the initialized instance variables, making the process dynamic.
        """
        table_name = self.source_catalog + '.' + self.source_schema + '.' + self.source_table_name
        self.daily_demand_feature_table = spark.read.table(table_name)
        #self.daily_demand_feature_table.show()

    def plot_bell_curve(self, output_variable):
        """
        Function to visualize the distribution of the output variable using Seaborn and Matplotlib.
        """
        pandas_df =  self.daily_demand_feature_table.select(F.col(output_variable)).toPandas()

        sns.histplot(
            pandas_df[output_variable],
            kde=True
        )

        plt.title("Distribution of Demand")
        plt.xlabel("Total Demand Quantity")
        plt.ylabel("Frequency")
        plt.show()

    def plot_coorelation_matrix(self, numeric_cols):
        """
        Function to visualize the correlation matrix of numerical features.
        """
        pandas_df = (
            self.daily_demand_feature_table
            .select(*[numeric_cols])
            .toPandas()
        )

        correlation_matrix = pandas_df.corr()

        plt.figure(figsize=(10, 7))

        sns.heatmap(
            correlation_matrix,
            annot=True,
            fmt=".2f",
            cmap="Blues"
        )

        plt.title("Correlation Matrix - Demand Features")
        plt.tight_layout()
        plt.show()

In [0]:
# numeric cols defincation for the correlation
numeric_cols = [
    "total_demand",
    "avg_unit_price",
    "total_inventory",
    "demand_lag_1",
    "demand_lag_7",
    "moving_avg_7",
]

In [0]:
### main part ###
data_vis = DataVisualization(source_catalog="supply_chain", source_schema="gold", source_table_name="daily_demand_features")

# read gold feature table
data_vis.read_feature_table()

# plot the output variable distribution
data_vis.plot_bell_curve(output_variable="total_demand")

# plot the correlation matrix for the numeric columns
data_vis.plot_coorelation_matrix(numeric_cols=numeric_cols)